# Full Package Dataset RC-PAA Paper Experiments

Dataset input: `NT230/data/full_package_dataset_balanced_10000.zip` on Google Drive.
Notebook n?y d?ng ?? ch?y test paper tr?n full package dataset hi?n t?i v? xu?t ?? output A-G.

- A ? Main results / Table 1: `main_results_table.csv`, `main_results_table.json`.
- B ? Ablation / Table 3: `ablation_cumulative.csv`, `ablation_leave_one_out.csv`.
- C ? Extra baselines: raw max, average pooling, majority voting trong `mil_baselines.csv`.
- D ? Threshold sensitivity: `threshold_sweep.csv`, `best_thresholds.json`, `threshold_cv_summary.csv`.
- E ? McNemar test: `mcnemar_tests.csv`, `mcnemar_tests.json`.
- F ? Audit/error JSON samples: `audit_samples.json`, `rcpaa_wrong_predictions.csv`, `rcpaa_false_positives.csv`, `rcpaa_false_negatives.csv`.
- G ? Weight table: `rcpaa_weight_table.csv`, `rcpaa_weight_table.json`.

Classifier v?n l? fine-tuned CodeBERT; c?c metric kh?ng g?i LLM. LLM rationale l? optional.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers==4.40.0 torch scikit-learn scipy pandas tqdm matplotlib ollama

In [ ]:
import os, shutil, sys, json, csv, math
from pathlib import Path
from collections import defaultdict, Counter

DRIVE_D1 = '/content/drive/My Drive/NT230/data/d1'
DRIVE_DATA = Path('/content/drive/My Drive/NT230/data')
DATASET_ZIP = DRIVE_DATA / 'full_package_dataset_balanced_10000.zip'
DATASET_NAME = 'full_package_dataset_balanced_10000'
DATASET_DIR = Path('/content') / DATASET_NAME
OUTPUT_DIR = DRIVE_DATA / 'results_full_package_dataset_rcpaa_paper_experiments'
REPO_DIR = Path('/content/NT230')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Experiment output:', OUTPUT_DIR)
print('K?t qu? cell n?y ch? ki?m tra d? li?u, model, repo path tr??c khi ch?y experiment.')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
!git clone --depth=1 https://github.com/khoilv2005/NT230.git /content/NT230
sys.path.insert(0, '/content/NT230/src')

MODEL_SRC = Path(DRIVE_D1) / 'saved_models/checkpoint-best-acc/model.bin'
MODEL_DST = Path('/content/saved_models/checkpoint-best-acc/model.bin')
MODEL_DST.parent.mkdir(parents=True, exist_ok=True)
assert MODEL_SRC.exists(), f'Missing model: {MODEL_SRC}'
shutil.copy(MODEL_SRC, MODEL_DST)
print('model.bin:', round(MODEL_DST.stat().st_size / 1e6), 'MB')

assert DATASET_ZIP.exists(), f'Missing dataset zip: {DATASET_ZIP}'
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
!unzip -q "{DATASET_ZIP}" -d /content

# Some zips contain a top-level folder; keep DATASET_DIR stable.
if not DATASET_DIR.exists():
    candidates = [p for p in Path('/content').glob('full_package_dataset*') if p.is_dir()]
    assert candidates, 'Dataset unzip completed but no full_package_dataset* folder found'
    DATASET_DIR = candidates[0]

for name in ['files.jsonl', 'packages.jsonl']:
    src = DATASET_DIR / name
    dst = Path('/content') / f'full_package_{name}'
    assert src.exists(), f'Missing dataset file: {src}'
    shutil.copy(src, dst)
    print(dst, sum(1 for _ in open(dst, encoding='utf-8')), 'records')
print('dataset_dir:', DATASET_DIR)


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')

In [ ]:
# Imports and helper functions
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import binomtest

from lamps.agents.classifier import ClassifierAgent, FileClassification
from lamps.agents.extractor import ExtractedFile
from lamps.agents.verdict import VerdictAgent
from lamps.agents.risk_calibrated_verdict import RiskCalibratedVerdictAgent
from lamps.evaluation.metrics import classification_report, format_report
from lamps.utils import read_jsonl

SEED = 230
BATCH_SIZE = 256
THRESHOLD = 0.72
RAW_THRESHOLD = 0.50


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def write_csv(path, rows, fieldnames=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = list(rows)
    if fieldnames is None:
        keys = []
        seen = set()
        for row in rows:
            for key in row.keys():
                if key not in seen:
                    seen.add(key)
                    keys.append(key)
        fieldnames = keys
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def report_row(name, y_true, y_pred, category):
    rep = classification_report(y_true, y_pred).to_dict()
    row = {
        'category': category,
        'method': name,
        'accuracy': rep['accuracy'],
        'balanced_accuracy': rep['balanced_accuracy'],
        'precision_malicious': rep['precision'],
        'recall_malicious': rep['recall'],
        'f1_malicious': rep['f1'],
        'benign_precision': rep['benign']['precision'],
        'benign_recall': rep['benign']['recall'],
        'benign_f1': rep['benign']['f1'],
        'TN': rep['confusion']['TN'],
        'FP': rep['confusion']['FP'],
        'FN': rep['confusion']['FN'],
        'TP': rep['confusion']['TP'],
        'n_samples': rep['n_samples'],
    }
    return row, rep


def mcnemar_exact(y_true, pred_a, pred_b, name_a, name_b):
    b = 0  # A wrong, B correct
    c = 0  # A correct, B wrong
    both_wrong = 0
    both_correct = 0
    for yt, pa, pb in zip(y_true, pred_a, pred_b):
        a_ok = int(pa) == int(yt)
        b_ok = int(pb) == int(yt)
        if (not a_ok) and b_ok:
            b += 1
        elif a_ok and (not b_ok):
            c += 1
        elif a_ok and b_ok:
            both_correct += 1
        else:
            both_wrong += 1
    n = b + c
    p_value = float(binomtest(min(b, c), n, p=0.5, alternative='two-sided').pvalue) if n else 1.0
    chi2_cc = float(((abs(b - c) - 1) ** 2) / n) if n else 0.0
    return {
        'method_a': name_a,
        'method_b': name_b,
        'a_wrong_b_correct': b,
        'a_correct_b_wrong': c,
        'both_correct': both_correct,
        'both_wrong': both_wrong,
        'discordant_pairs': n,
        'exact_p_value': p_value,
        'chi2_continuity_corrected': chi2_cc,
        'significant_at_0.05': bool(p_value < 0.05),
    }


def print_table(rows, title):
    print('\n' + title)
    cols = ['method', 'accuracy', 'balanced_accuracy', 'precision_malicious', 'recall_malicious', 'f1_malicious', 'TN', 'FP', 'FN', 'TP']
    df = pd.DataFrame(rows)[cols]
    display(df)
    return df

In [ ]:
# Load full package dataset records and write dataset clarification
print('Cell n?y load full_package_dataset_balanced_10000 v? ghi r? scope dataset.')
file_records = list(read_jsonl(Path('/content/full_package_files.jsonl')))
package_records = list(read_jsonl(Path('/content/full_package_packages.jsonl')))
package_labels = Counter(int(r.get('label', -1)) for r in package_records)
file_labels = Counter(int(r.get('target', r.get('label', -1))) for r in file_records)

print('Files:', len(file_records))
print('Packages:', len(package_records))
print('Package labels:', package_labels)
print('File labels:', file_labels)

clarification = (
    '# Dataset clarification\n\n'
    'This experiment uses full_package_dataset_balanced_10000.zip from Google Drive, not the repository D2 dataset.\n\n'
    'Current full package dataset summary:\n\n'
    f'- Packages: {len(package_records)}\n'
    f'- Files before extractor filter: {len(file_records)}\n'
    f'- Package labels: {dict(package_labels)}\n'
    f'- File labels in records: {dict(file_labels)}\n\n'
    'Evaluation unit:\n\n'
    '- Main metrics are package-level.\n'
    '- File-level labels are propagated from package labels in the current D2 build and are diagnostic only.\n'
    '- Paper claims should say: RC-PAA improves package-level aggregation under the current full-package balanced dataset setting.\n'
    '- Do not claim exact replication of original paper D2; this is a balanced full-package dataset built for evaluation.\n'
)
(OUTPUT_DIR / 'dataset_clarification.md').write_text(clarification, encoding='utf-8')
print(clarification)


In [ ]:
# Extractor Agent - same filter as existing evaluate_d2 notebooks.
print('Cell n?y l?c file gi?ng evaluate_d2: b? noisy docs/tests/vendor ?? gi? file Python relevant cho Classifier Agent.')
NOISY = {'tests', 'test', 'testing', 'docs', 'doc', 'examples', '_vendor', 'vendor'}

def is_relevant(path):
    parts = str(path).lower().replace('\\', '/').split('/')
    return not any(p in NOISY for p in parts)

filtered = [r for r in file_records if is_relevant(r.get('path', ''))]
files = [
    ExtractedFile(
        package=str(r['package']),
        path=Path('<memory>'),
        rel_path=str(r.get('path', '')),
        source=str(r['func']),
    )
    for r in filtered
]

print('Filtered files:', len(filtered), '/', len(file_records))
print('Filtered labels diagnostic:', Counter(int(r.get('target', -1)) for r in filtered))

In [ ]:
# Classifier Agent - CodeBERT per-file with cache
print('Cell n?y ch?y CodeBERT m?t l?n. N?u ?? c? cache th? load l?i ?? c?c experiment ph?a sau kh?ng classify l?i.')
CLASSIFICATION_CACHE = OUTPUT_DIR / 'file_classifications.jsonl'

if CLASSIFICATION_CACHE.exists():
    print('Loading cached classifications:', CLASSIFICATION_CACHE)
    classification_rows = list(read_jsonl(CLASSIFICATION_CACHE))
    classifications = [
        FileClassification(
            package=str(r['package']),
            rel_path=str(r['rel_path']),
            label=str(r['label']),
            target=int(r['target']),
            score=float(r['score']),
        )
        for r in classification_rows
    ]
else:
    classifier = ClassifierAgent(checkpoint=MODEL_DST, device='cuda' if torch.cuda.is_available() else 'cpu', batch_size=BATCH_SIZE)
    classifications = []
    for start in tqdm(range(0, len(files), BATCH_SIZE), desc='CodeBERT batches'):
        batch = files[start:start + BATCH_SIZE]
        classifications.extend(classifier.classify_files(batch))
    classification_rows = [
        {'package': c.package, 'rel_path': c.rel_path, 'label': c.label, 'target': int(c.target), 'score': float(c.score)}
        for c in classifications
    ]
    write_jsonl(CLASSIFICATION_CACHE, classification_rows)
    print('Saved cache:', CLASSIFICATION_CACHE)

assert len(classifications) == len(files), (len(classifications), len(files))
print('Classifications:', len(classifications))
print('Predicted file labels diagnostic:', Counter(int(c.target) for c in classifications))

In [ ]:
# Build package index
cls_by_pkg = defaultdict(list)
files_by_pkg = defaultdict(list)
for cls, extracted in zip(classifications, files):
    cls_by_pkg[cls.package].append(cls)
    files_by_pkg[extracted.package].append(extracted)

packages = [str(p['package']) for p in package_records]
y_true = [int(p['label']) for p in package_records]
print('Packages with classifications:', sum(1 for p in packages if cls_by_pkg.get(p)))
print('Total packages:', len(packages))

In [ ]:
# MIL baselines and RC-PAA
print('Cell n?y so s?nh c?c aggregation baseline b?t bu?c cho paper.')

def predict_original_any_file(preds):
    return 1 if any(int(p.target) == 1 for p in preds) else 0


def predict_raw_max(preds, threshold=RAW_THRESHOLD):
    return 1 if max([float(p.score) for p in preds], default=0.0) >= threshold else 0


def predict_average_pooling(preds, threshold=RAW_THRESHOLD):
    if not preds:
        return 0
    return 1 if (sum(float(p.score) for p in preds) / len(preds)) >= threshold else 0


def predict_majority_voting(preds):
    if not preds:
        return 0
    return 1 if (sum(int(p.target) for p in preds) / len(preds)) > 0.5 else 0

rcpaa_agent = RiskCalibratedVerdictAgent(llm=None, package_threshold=THRESHOLD)
rcpaa_structured_agent = RiskCalibratedVerdictAgent(llm=None, package_threshold=THRESHOLD, )

predictions_by_method = {
    'original_any_file': [],
    'raw_max_pooling_0.50': [],
    'average_pooling_0.50': [],
    'majority_voting': [],
    'rcpaa_0.72': [],
    'rcpaa_structured_0.72': [],
}
package_score_rows = []
risk_rows_structured = []

for pkg in package_records:
    name = str(pkg['package'])
    preds = cls_by_pkg.get(name, [])
    pkg_files = files_by_pkg.get(name, [])

    original = predict_original_any_file(preds)
    raw_max = max([float(p.score) for p in preds], default=0.0)
    average = (sum(float(p.score) for p in preds) / len(preds)) if preds else 0.0
    majority_ratio = (sum(int(p.target) for p in preds) / len(preds)) if preds else 0.0
    rcpaa_verdict = rcpaa_agent.aggregate(name, preds, pkg_files)
    top_risk = sorted(rcpaa_agent.last_file_risks, key=lambda r: r.calibrated_score, reverse=True)
    rcpaa_score = top_risk[0].calibrated_score if top_risk else 0.0
    trigger_file = top_risk[0].rel_path if top_risk else ''

    rcpaa_structured_verdict = rcpaa_structured_agent.aggregate(name, preds, pkg_files)
    top_risk_structured = sorted(rcpaa_structured_agent.last_file_risks, key=lambda r: r.calibrated_score, reverse=True)
    rcpaa_structured_score = top_risk_structured[0].calibrated_score if top_risk_structured else 0.0
    trigger_file_structured = top_risk_structured[0].rel_path if top_risk_structured else ''

    predictions_by_method['original_any_file'].append(original)
    predictions_by_method['raw_max_pooling_0.50'].append(1 if raw_max >= RAW_THRESHOLD else 0)
    predictions_by_method['average_pooling_0.50'].append(1 if average >= RAW_THRESHOLD else 0)
    predictions_by_method['majority_voting'].append(1 if majority_ratio > 0.5 else 0)
    predictions_by_method['rcpaa_0.72'].append(int(rcpaa_verdict.target))
    predictions_by_method['rcpaa_structured_0.72'].append(int(rcpaa_structured_verdict.target))

    package_score_rows.append({
        'package': name,
        'target': int(pkg['label']),
        'n_files_total': int(pkg.get('n_files', 0)),
        'n_files_after_filter': len(preds),
        'raw_max_score': raw_max,
        'average_score': average,
        'majority_ratio': majority_ratio,
        'rcpaa_score': rcpaa_score,
        'rcpaa_prediction': int(rcpaa_verdict.target),
        'rcpaa_structured_score': rcpaa_structured_score,
        'rcpaa_structured_prediction': int(rcpaa_structured_verdict.target),
        'trigger_file': trigger_file,
        'trigger_file_structured': trigger_file_structured,
        'structured_outcome': json.dumps(rcpaa_structured_verdict.structured_outcome, ensure_ascii=False),
        'top_risk_files': json.dumps([
            {'path': r.rel_path, 'base_score': round(r.base_score, 4), 'calibrated_score': round(r.calibrated_score, 4), 'reasons': r.reasons}
            for r in top_risk[:5]
        ], ensure_ascii=False),
    })
    for r in rcpaa_agent.risk_rows():
        risk_rows_structured.append(r)

mil_rows = []
reports = {}
for method, pred in predictions_by_method.items():
    row, rep = report_row(method, y_true, pred, 'mil_baseline')
    mil_rows.append(row)
    reports[method] = rep

print_table(mil_rows, 'MIL baseline comparison')
write_csv(OUTPUT_DIR / 'mil_baselines.csv', mil_rows)
write_json(OUTPUT_DIR / 'mil_baselines.json', {'rows': mil_rows, 'reports': reports})
write_csv(OUTPUT_DIR / 'package_scores.csv', package_score_rows)
write_jsonl(OUTPUT_DIR / 'risk_file_scores.jsonl', risk_rows_structured)

In [ ]:
# Optional LLM rationale for RC-PAA structured
print('Cell này optional. USE_LLM_RATIONALE=True nếu muốn gọi Ollama Cloud tạo rationale theo structured outcome.')
USE_LLM_RATIONALE = False
LLM_RATIONALE_LIMIT = 5

if USE_LLM_RATIONALE:
    import os, re
    from lamps.llms.ollama_client import OllamaClient

    env_path = DRIVE_DATA / '.env'
    if env_path.exists():
        for line in env_path.read_text(encoding='utf-8').splitlines():
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            m = re.match(r'^\$env:(\w+)\s*=\s*["\']?([^"\']+)["\']?', line)
            if m:
                os.environ.setdefault(m.group(1), m.group(2).strip())
            elif '=' in line:
                k, _, v = line.partition('=')
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

    llm = OllamaClient(model=os.getenv('OLLAMA_MODEL', 'deepseek-v4-flash:cloud'))
    llm_agent = RiskCalibratedVerdictAgent(llm=llm, package_threshold=THRESHOLD, )
    rationale_rows = []
    selected = package_score_rows[:LLM_RATIONALE_LIMIT]
    for row in selected:
        name = row['package']
        verdict = llm_agent.aggregate(name, cls_by_pkg.get(name, []), files_by_pkg.get(name, []))
        rationale_rows.append(verdict.to_dict())
        print(name, '=>', verdict.label, 'confidence=', round(verdict.confidence, 4))
        print(verdict.rationale)
    write_jsonl(OUTPUT_DIR / 'rcpaa_structured_llm_rationales.jsonl', rationale_rows)
else:
    print('Skipped LLM rationale. Metric experiments do not require LLM calls.')


In [ ]:
# RC-PAA ablation class
print('Cell n?y ??nh ngh?a ablation agent ?? b?t/t?t t?ng modifier m? kh?ng s?a source repo.')

class RCPAAAblationAgent(RiskCalibratedVerdictAgent):
    def __init__(
        self,
        *,
        enable_critical=True,
        enable_behavior=True,
        enable_import=True,
        enable_context=True,
        enable_low_confidence=True,
        enable_large_package=True,
        enable_generated=True,
        enable_small_rescue=True,
        package_threshold=THRESHOLD,
    ):
        super().__init__(
            llm=None,
            package_threshold=package_threshold,
            low_confidence_penalty=0.14 if enable_low_confidence else 0.0,
            large_package_penalty=0.10 if enable_large_package else 0.0,
            generated_penalty=0.28 if enable_generated else 0.0,
            small_package_behavior_bonus=0.28 if enable_small_rescue else 0.0,
        )
        self.enable_critical = enable_critical
        self.enable_behavior = enable_behavior
        self.enable_import = enable_import
        self.enable_context = enable_context
        self.enable_large_package = enable_large_package
        self.enable_generated = enable_generated
        self.enable_small_rescue = enable_small_rescue

    def _critical_role_bonus(self, rel_path, source):
        if not self.enable_critical:
            return 0.0
        return super()._critical_role_bonus(rel_path, source)

    def _critical_import_targets(self, source_by_path):
        if not self.enable_import:
            return set()
        return super()._critical_import_targets(source_by_path)

    def _behavior_score(self, source):
        if not self.enable_behavior:
            return 0.0
        return super()._behavior_score(source)

    def _context_penalty(self, rel_path, is_critical, imported_by_critical):
        if not self.enable_context:
            return 0.0, []
        return super()._context_penalty(rel_path, is_critical, imported_by_critical)

    def _large_package_penalty(self, rel_path, n_files, is_critical, imported_by_critical):
        if not self.enable_large_package:
            return 0.0
        return super()._large_package_penalty(rel_path, n_files, is_critical, imported_by_critical)

    def _generated_resource_penalty(self, rel_path, is_critical, imported_by_critical):
        if not self.enable_generated:
            return 0.0, []
        return super()._generated_resource_penalty(rel_path, is_critical, imported_by_critical)

    def _small_package_behavior_rescue(self, rel_path, source, behavior, n_files):
        if not (self.enable_small_rescue and self.enable_behavior):
            return False
        return super()._small_package_behavior_rescue(rel_path, source, behavior, n_files)


def run_rcpaa_agent(agent, name):
    preds_out = []
    score_out = []
    for pkg in package_records:
        package_name = str(pkg['package'])
        verdict = agent.aggregate(package_name, cls_by_pkg.get(package_name, []), files_by_pkg.get(package_name, []))
        top = sorted(agent.last_file_risks, key=lambda r: r.calibrated_score, reverse=True)
        preds_out.append(int(verdict.target))
        score_out.append(float(top[0].calibrated_score if top else 0.0))
    row, rep = report_row(name, y_true, preds_out, 'ablation')
    return row, rep, preds_out, score_out

In [ ]:
# Cumulative ablation study
print('Cell n?y ch?y cumulative ablation: th?m d?n t?ng nh?m modifier v?o CodeBERT max-pooling.')

ablation_configs = [
    ('codebert_max_only', dict(enable_critical=False, enable_behavior=False, enable_import=False, enable_context=False, enable_low_confidence=False, enable_large_package=False, enable_generated=False, enable_small_rescue=False, package_threshold=RAW_THRESHOLD)),
    ('+critical_file_bonus', dict(enable_critical=True, enable_behavior=False, enable_import=False, enable_context=False, enable_low_confidence=False, enable_large_package=False, enable_generated=False, enable_small_rescue=False, package_threshold=THRESHOLD)),
    ('+behavior_bonus', dict(enable_critical=True, enable_behavior=True, enable_import=False, enable_context=False, enable_low_confidence=False, enable_large_package=False, enable_generated=False, enable_small_rescue=True, package_threshold=THRESHOLD)),
    ('+cross_file_import_bonus', dict(enable_critical=True, enable_behavior=True, enable_import=True, enable_context=False, enable_low_confidence=False, enable_large_package=False, enable_generated=False, enable_small_rescue=True, package_threshold=THRESHOLD)),
    ('+context_low_conf_penalty', dict(enable_critical=True, enable_behavior=True, enable_import=True, enable_context=True, enable_low_confidence=True, enable_large_package=True, enable_generated=True, enable_small_rescue=True, package_threshold=THRESHOLD)),
]

ablation_rows = []
ablation_reports = {}
ablation_predictions = {}
ablation_scores = {}
for name, kwargs in ablation_configs:
    agent = RCPAAAblationAgent(**kwargs)
    row, rep, pred, scores = run_rcpaa_agent(agent, name)
    ablation_rows.append(row)
    ablation_reports[name] = rep
    ablation_predictions[name] = pred
    ablation_scores[name] = scores

print_table(ablation_rows, 'Cumulative ablation')
write_csv(OUTPUT_DIR / 'ablation_cumulative.csv', ablation_rows)
write_json(OUTPUT_DIR / 'ablation_cumulative.json', {'rows': ablation_rows, 'reports': ablation_reports})

In [ ]:
# Leave-one-out ablation
print('Cell n?y ch?y leave-one-out ablation: b? t?ng modifier kh?i full RC-PAA ?? xem m?t g?.')

loo_configs = [
    ('full_rcpaa', dict()),
    ('without_critical', dict(enable_critical=False)),
    ('without_behavior', dict(enable_behavior=False, enable_small_rescue=False)),
    ('without_import', dict(enable_import=False)),
    ('without_context_penalty', dict(enable_context=False)),
    ('without_low_confidence_penalty', dict(enable_low_confidence=False)),
    ('without_large_package_penalty', dict(enable_large_package=False)),
    ('without_generated_penalty', dict(enable_generated=False)),
    ('without_small_rescue', dict(enable_small_rescue=False)),
]

loo_rows = []
loo_reports = {}
loo_predictions = {}
for name, kwargs in loo_configs:
    agent = RCPAAAblationAgent(**kwargs)
    row, rep, pred, scores = run_rcpaa_agent(agent, name)
    loo_rows.append(row)
    loo_reports[name] = rep
    loo_predictions[name] = pred

full_f1 = next(r['f1_malicious'] for r in loo_rows if r['method'] == 'full_rcpaa')
full_bal = next(r['balanced_accuracy'] for r in loo_rows if r['method'] == 'full_rcpaa')
for row in loo_rows:
    row['delta_f1_vs_full'] = row['f1_malicious'] - full_f1
    row['delta_balanced_accuracy_vs_full'] = row['balanced_accuracy'] - full_bal

print_table(loo_rows, 'Leave-one-out ablation')
write_csv(OUTPUT_DIR / 'ablation_leave_one_out.csv', loo_rows)
write_json(OUTPUT_DIR / 'ablation_leave_one_out.json', {'rows': loo_rows, 'reports': loo_reports})

In [ ]:
# Threshold sweep and threshold validation
print('Cell n?y sweep threshold cho raw max, average pooling, RC-PAA ? RC-PAA structured. Best threshold ch?n theo F1 v? balanced accuracy.')
thresholds = [round(x / 100, 2) for x in range(5, 96, 5)] + [0.72]
thresholds = sorted(set(thresholds))

score_maps = {
    'raw_max_pooling': [row['raw_max_score'] for row in package_score_rows],
    'average_pooling': [row['average_score'] for row in package_score_rows],
    'rcpaa': [row['rcpaa_score'] for row in package_score_rows],
    'rcpaa_structured': [row['rcpaa_structured_score'] for row in package_score_rows],
}

threshold_rows = []
for score_name, scores in score_maps.items():
    for th in thresholds:
        pred = [1 if float(s) >= th else 0 for s in scores]
        row, rep = report_row(f'{score_name}@{th:.2f}', y_true, pred, 'threshold_sweep')
        row['score_name'] = score_name
        row['threshold'] = th
        threshold_rows.append(row)

best_by_f1 = {}
best_by_bal = {}
for score_name in score_maps:
    rows = [r for r in threshold_rows if r['score_name'] == score_name]
    best_by_f1[score_name] = max(rows, key=lambda r: (r['f1_malicious'], r['balanced_accuracy'], r['precision_malicious']))
    best_by_bal[score_name] = max(rows, key=lambda r: (r['balanced_accuracy'], r['f1_malicious'], r['precision_malicious']))

print('Best threshold by F1:')
display(pd.DataFrame(best_by_f1.values())[['score_name', 'threshold', 'accuracy', 'balanced_accuracy', 'precision_malicious', 'recall_malicious', 'f1_malicious', 'TN', 'FP', 'FN', 'TP']])
print('Best threshold by balanced accuracy:')
display(pd.DataFrame(best_by_bal.values())[['score_name', 'threshold', 'accuracy', 'balanced_accuracy', 'precision_malicious', 'recall_malicious', 'f1_malicious', 'TN', 'FP', 'FN', 'TP']])

write_csv(OUTPUT_DIR / 'threshold_sweep.csv', threshold_rows)
write_jsonl(OUTPUT_DIR / 'threshold_sweep.jsonl', threshold_rows)
write_json(OUTPUT_DIR / 'best_thresholds.json', {'by_f1': best_by_f1, 'by_balanced_accuracy': best_by_bal})

In [ ]:
# 5-fold threshold validation
print('Cell n?y l?m 5-fold CV package-level ?? ch?n threshold tr?n train fold r?i ??nh gi? tr?n validation fold.')
from sklearn.model_selection import StratifiedKFold

cv_threshold_rows = []
cv_summary_rows = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
threshold_grid = thresholds

for score_name, scores in score_maps.items():
    scores = list(map(float, scores))
    fold_rows = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(scores, y_true), start=1):
        train_scores = [scores[i] for i in train_idx]
        train_true = [y_true[i] for i in train_idx]
        val_scores = [scores[i] for i in val_idx]
        val_true = [y_true[i] for i in val_idx]

        train_candidates = []
        for th in threshold_grid:
            train_pred = [1 if s >= th else 0 for s in train_scores]
            row, rep = report_row(f'{score_name}_train_fold{fold}@{th:.2f}', train_true, train_pred, 'threshold_cv_train')
            row['score_name'] = score_name
            row['fold'] = fold
            row['threshold'] = th
            train_candidates.append(row)

        best_train = max(train_candidates, key=lambda r: (r['f1_malicious'], r['balanced_accuracy'], r['precision_malicious']))
        th = float(best_train['threshold'])
        val_pred = [1 if s >= th else 0 for s in val_scores]
        val_row, val_rep = report_row(f'{score_name}_val_fold{fold}@{th:.2f}', val_true, val_pred, 'threshold_cv_val')
        val_row['score_name'] = score_name
        val_row['fold'] = fold
        val_row['threshold'] = th
        val_row['selected_by'] = 'train_f1'
        val_row['train_f1'] = best_train['f1_malicious']
        val_row['train_balanced_accuracy'] = best_train['balanced_accuracy']
        fold_rows.append(val_row)
        cv_threshold_rows.extend(train_candidates)
        cv_threshold_rows.append(val_row)

    mean_row = {'score_name': score_name, 'category': 'threshold_cv_mean', 'method': f'{score_name}_5fold_cv_mean'}
    metric_cols = ['accuracy', 'balanced_accuracy', 'precision_malicious', 'recall_malicious', 'f1_malicious', 'TN', 'FP', 'FN', 'TP']
    for col in metric_cols:
        mean_row[col] = float(sum(float(r[col]) for r in fold_rows) / len(fold_rows))
    mean_row['thresholds_selected'] = ','.join(f"{r['threshold']:.2f}" for r in fold_rows)
    cv_summary_rows.append(mean_row)

print('5-fold CV mean:')
display(pd.DataFrame(cv_summary_rows))
write_csv(OUTPUT_DIR / 'threshold_cv_rows.csv', cv_threshold_rows)
write_csv(OUTPUT_DIR / 'threshold_cv_summary.csv', cv_summary_rows)
write_json(OUTPUT_DIR / 'threshold_cv.json', {'rows': cv_threshold_rows, 'summary': cv_summary_rows})

In [ ]:
# McNemar tests
print('Cell n?y ch?y McNemar exact test tr?n paired package predictions.')
mcnemar_rows = []
base_methods = ['original_any_file', 'raw_max_pooling_0.50', 'average_pooling_0.50', 'majority_voting', 'rcpaa_0.72']
for base in base_methods:
    mcnemar_rows.append(mcnemar_exact(y_true, predictions_by_method[base], predictions_by_method['rcpaa_structured_0.72'], base, 'rcpaa_structured_0.72'))
for base in ['codebert_max_only', '+critical_file_bonus', '+behavior_bonus', '+cross_file_import_bonus']:
    mcnemar_rows.append(mcnemar_exact(y_true, ablation_predictions[base], ablation_predictions['+context_low_conf_penalty'], base, 'full_ablation_rcpaa'))

display(pd.DataFrame(mcnemar_rows))
write_csv(OUTPUT_DIR / 'mcnemar_tests.csv', mcnemar_rows)
write_json(OUTPUT_DIR / 'mcnemar_tests.json', mcnemar_rows)

In [ ]:
# F ? Audit JSON samples and FP/FN error lists
print('Cell n?y xu?t audit JSON m?u + danh s?ch FP/FN cho Error Analysis.')

rcpaa_pred = predictions_by_method['rcpaa_structured_0.72']
wrong_rows = []
fp_rows = []
fn_rows = []
for row, yt, yp in zip(package_score_rows, y_true, rcpaa_pred):
    item = dict(row)
    item['predicted'] = int(yp)
    item['error_type'] = 'correct' if int(yt) == int(yp) else ('FP' if int(yt) == 0 else 'FN')
    try:
        item['structured_outcome_obj'] = json.loads(item.get('structured_outcome') or '{}')
    except Exception:
        item['structured_outcome_obj'] = {}
    if item['error_type'] != 'correct':
        wrong_rows.append(item)
        if item['error_type'] == 'FP':
            fp_rows.append(item)
        else:
            fn_rows.append(item)

wrong_rows_sorted = sorted(wrong_rows, key=lambda r: (r['error_type'], -float(r['rcpaa_structured_score']), r['package']))
fp_rows_sorted = sorted(fp_rows, key=lambda r: (-float(r['rcpaa_structured_score']), r['package']))
fn_rows_sorted = sorted(fn_rows, key=lambda r: (-float(r['raw_max_score']), r['package']))

correct_malicious = [dict(row) for row, yt, yp in zip(package_score_rows, y_true, rcpaa_pred) if int(yt) == 1 and int(yp) == 1]
correct_benign = [dict(row) for row, yt, yp in zip(package_score_rows, y_true, rcpaa_pred) if int(yt) == 0 and int(yp) == 0]
correct_malicious = sorted(correct_malicious, key=lambda r: -float(r['rcpaa_structured_score']))
correct_benign = sorted(correct_benign, key=lambda r: -float(r['rcpaa_structured_score']))

audit_samples = {
    'high_confidence_true_positive': correct_malicious[0] if correct_malicious else None,
    'high_residual_true_negative': correct_benign[0] if correct_benign else None,
    'false_positive_example': fp_rows_sorted[0] if fp_rows_sorted else None,
    'false_negative_example': fn_rows_sorted[0] if fn_rows_sorted else None,
    'counts': {
        'wrong': len(wrong_rows_sorted),
        'false_positives': len(fp_rows_sorted),
        'false_negatives': len(fn_rows_sorted),
    },
}

audit_fieldnames = ['package', 'target', 'predicted', 'error_type', 'n_files_total', 'n_files_after_filter', 'raw_max_score', 'average_score', 'majority_ratio', 'rcpaa_structured_score', 'trigger_file_structured', 'top_risk_files', 'structured_outcome']
write_csv(OUTPUT_DIR / 'rcpaa_wrong_predictions.csv', wrong_rows_sorted, audit_fieldnames)
write_csv(OUTPUT_DIR / 'rcpaa_false_positives.csv', fp_rows_sorted, audit_fieldnames)
write_csv(OUTPUT_DIR / 'rcpaa_false_negatives.csv', fn_rows_sorted, audit_fieldnames)
write_jsonl(OUTPUT_DIR / 'rcpaa_wrong_predictions.jsonl', wrong_rows_sorted)
write_jsonl(OUTPUT_DIR / 'rcpaa_false_positives.jsonl', fp_rows_sorted)
write_jsonl(OUTPUT_DIR / 'rcpaa_false_negatives.jsonl', fn_rows_sorted)
write_json(OUTPUT_DIR / 'audit_samples.json', audit_samples)

print('Wrong/FP/FN:', len(wrong_rows_sorted), len(fp_rows_sorted), len(fn_rows_sorted))
print('Audit samples saved:', OUTPUT_DIR / 'audit_samples.json')
display(pd.DataFrame(wrong_rows_sorted)[audit_fieldnames].head(10) if wrong_rows_sorted else pd.DataFrame())


In [ ]:
# G ? RC-PAA weight table for reproducibility
print('Cell n?y xu?t to?n b? weight/threshold RC-PAA ?? reproduce c?ng th?c.')

weight_agent = RiskCalibratedVerdictAgent(llm=None, package_threshold=THRESHOLD)
weight_rows = [
    {'group': 'threshold', 'symbol': 'tau', 'component': 'package_threshold', 'value': weight_agent.package_threshold, 'direction': 'decision', 'code_location': 'RiskCalibratedVerdictAgent.package_threshold'},
    {'group': 'threshold', 'symbol': 'tau_raw', 'component': 'raw_codebert_threshold', 'value': RAW_THRESHOLD, 'direction': 'decision', 'code_location': 'RAW_THRESHOLD'},
    {'group': 'risk_bonus', 'symbol': 'alpha_setup', 'component': 'setup.py critical role', 'value': 0.26, 'direction': '+', 'code_location': '_critical_role_bonus'},
    {'group': 'risk_bonus', 'symbol': 'alpha_install', 'component': 'install/build/post_install critical role', 'value': 0.24, 'direction': '+', 'code_location': '_critical_role_bonus'},
    {'group': 'risk_bonus', 'symbol': 'alpha_main', 'component': '__main__.py critical role', 'value': 0.18, 'direction': '+', 'code_location': '_critical_role_bonus'},
    {'group': 'risk_bonus', 'symbol': 'alpha_init', 'component': '__init__.py critical role', 'value': 0.08, 'direction': '+', 'code_location': '_critical_role_bonus'},
    {'group': 'risk_bonus', 'symbol': 'alpha_other_critical', 'component': 'other setup/install path', 'value': min(weight_agent.critical_bonus, 0.16), 'direction': '+', 'code_location': '_critical_role_bonus'},
    {'group': 'risk_bonus', 'symbol': 'alpha_import', 'component': 'imported by entrypoint', 'value': weight_agent.import_bonus, 'direction': '+', 'code_location': 'import_bonus'},
    {'group': 'risk_bonus', 'symbol': 'alpha_behavior', 'component': 'behavior score multiplier', 'value': weight_agent.behavior_bonus, 'direction': '+', 'code_location': 'behavior_bonus * behavior_score'},
    {'group': 'risk_bonus', 'symbol': 'alpha_small_rescue', 'component': 'small package behavior rescue', 'value': weight_agent.small_package_behavior_bonus, 'direction': '+', 'code_location': 'small_package_behavior_bonus'},
    {'group': 'benign_penalty', 'symbol': 'beta_low_conf', 'component': 'low confidence non-critical non-imported file', 'value': weight_agent.low_confidence_penalty, 'direction': '-', 'code_location': 'low_confidence_penalty'},
    {'group': 'benign_penalty', 'symbol': 'beta_docs_tests', 'component': 'docs/examples/tests/benchmarks path', 'value': weight_agent.docs_examples_penalty, 'direction': '-', 'code_location': 'docs_examples_penalty'},
    {'group': 'benign_penalty', 'symbol': 'beta_tooling', 'component': 'tools/scripts/dev/demo/sample path', 'value': weight_agent.tooling_penalty, 'direction': '-', 'code_location': 'tooling_penalty'},
    {'group': 'benign_penalty', 'symbol': 'beta_generated', 'component': 'generated/resource/fixture file', 'value': weight_agent.generated_penalty, 'direction': '-', 'code_location': 'generated_penalty'},
    {'group': 'benign_penalty', 'symbol': 'beta_generated_critical', 'component': 'generated/resource if critical or imported', 'value': weight_agent.generated_penalty * 0.5, 'direction': '-', 'code_location': '_generated_resource_penalty'},
    {'group': 'benign_penalty', 'symbol': 'beta_large_50', 'component': 'large package non-critical n_files >= 50', 'value': weight_agent.large_package_penalty, 'direction': '-', 'code_location': '_large_package_penalty'},
    {'group': 'benign_penalty', 'symbol': 'beta_large_100', 'component': 'large package non-critical n_files >= 100', 'value': 0.14, 'direction': '-', 'code_location': '_large_package_penalty'},
    {'group': 'benign_penalty', 'symbol': 'beta_large_250', 'component': 'large package non-critical n_files >= 250', 'value': 0.18, 'direction': '-', 'code_location': '_large_package_penalty'},
    {'group': 'benign_penalty', 'symbol': 'beta_large_500', 'component': 'large package non-critical n_files >= 500', 'value': 0.22, 'direction': '-', 'code_location': '_large_package_penalty'},
]
behavior_indicator_rows = [
    ('gamma_pair', 'base64 + exec/eval/subprocess or network + exec/socket connect', 0.55),
    ('gamma_subprocess', 'subprocess.Popen/call/run', 0.25),
    ('gamma_os_popen', 'os.popen', 0.25),
    ('gamma_os_system', 'os.system', 0.22),
    ('gamma_exec', 'exec', 0.30),
    ('gamma_eval', 'eval', 0.24),
    ('gamma_compile', 'compile', 0.10),
    ('gamma_network', 'requests/urllib get/post/urlopen', 0.16),
    ('gamma_socket_mail', 'socket/ftplib/smtplib', 0.14),
    ('gamma_secret_terms', 'token/password/secret/api_key/webhook', 0.08),
    ('gamma_clear_penalty', 'os.system clear/cls benign terminal clear', -0.18),
    ('gamma_data_uri_penalty', 'data:image or base64 pdf literal', -0.12),
]
for symbol, component, value in behavior_indicator_rows:
    weight_rows.append({'group': 'behavior_indicator', 'symbol': symbol, 'component': component, 'value': value, 'direction': '+' if value >= 0 else '-', 'code_location': '_behavior_score'})

write_csv(OUTPUT_DIR / 'rcpaa_weight_table.csv', weight_rows)
write_json(OUTPUT_DIR / 'rcpaa_weight_table.json', weight_rows)
display(pd.DataFrame(weight_rows))


In [ ]:
# Summary tables, figures, and A-G output manifest
print('Cell n?y xu?t b?ng t?ng h?p cu?i + manifest A-G cho paper outputs.')
summary_rows = []
summary_rows.extend(mil_rows)
summary_rows.extend(ablation_rows)
summary_rows.extend(loo_rows)
summary_rows.extend(cv_summary_rows)

main_methods = ['original_any_file', 'rcpaa_structured_0.72']
baseline_methods = ['original_any_file', 'raw_max_pooling_0.50', 'average_pooling_0.50', 'majority_voting', 'rcpaa_structured_0.72']
main_results_rows = [r for r in mil_rows if r['method'] in main_methods]
baseline_results_rows = [r for r in mil_rows if r['method'] in baseline_methods]

write_csv(OUTPUT_DIR / 'main_results_table.csv', main_results_rows)
write_json(OUTPUT_DIR / 'main_results_table.json', main_results_rows)
write_csv(OUTPUT_DIR / 'baseline_results_table.csv', baseline_results_rows)
write_json(OUTPUT_DIR / 'baseline_results_table.json', baseline_results_rows)
write_csv(OUTPUT_DIR / 'summary_table.csv', summary_rows)

output_manifest = {
    'A_main_results': ['main_results_table.csv', 'main_results_table.json', 'mil_baselines.csv'],
    'B_ablation': ['ablation_cumulative.csv', 'ablation_cumulative.json', 'ablation_leave_one_out.csv', 'ablation_leave_one_out.json'],
    'C_extra_baselines': ['baseline_results_table.csv', 'baseline_results_table.json', 'mil_baselines.csv'],
    'D_threshold_sensitivity': ['threshold_sweep.csv', 'threshold_sweep.jsonl', 'best_thresholds.json', 'threshold_cv_summary.csv', 'threshold_cv.json'],
    'E_mcnemar': ['mcnemar_tests.csv', 'mcnemar_tests.json'],
    'F_audit_and_errors': ['audit_samples.json', 'rcpaa_wrong_predictions.csv', 'rcpaa_false_positives.csv', 'rcpaa_false_negatives.csv'],
    'G_weight_table': ['rcpaa_weight_table.csv', 'rcpaa_weight_table.json'],
}
output_checklist = {
    group: {'complete': all((OUTPUT_DIR / name).exists() for name in files), 'files': files}
    for group, files in output_manifest.items()
}

write_json(OUTPUT_DIR / 'output_manifest.json', output_manifest)
write_json(OUTPUT_DIR / 'output_checklist.json', output_checklist)
write_json(OUTPUT_DIR / 'summary.json', {
    'n_packages': len(package_records),
    'n_files_before_filter': len(file_records),
    'n_filtered_files': len(filtered),
    'package_labels': dict(package_labels),
    'threshold_fixed': THRESHOLD,
    'raw_threshold_fixed': RAW_THRESHOLD,
    'main_results': main_results_rows,
    'mil_baselines': mil_rows,
    'ablation_cumulative': ablation_rows,
    'ablation_leave_one_out': loo_rows,
    'best_thresholds': {'by_f1': best_by_f1, 'by_balanced_accuracy': best_by_bal},
    'threshold_cv_summary': cv_summary_rows,
    'mcnemar_tests': mcnemar_rows,
    'audit_counts': audit_samples['counts'] if 'audit_samples' in globals() else {},
    'output_checklist': output_checklist,
    'output': str(OUTPUT_DIR),
})

main_df = pd.DataFrame(baseline_results_rows).set_index('method').loc[baseline_methods].reset_index()
plt.figure(figsize=(10, 5))
x = range(len(main_df))
plt.bar([i - 0.18 for i in x], main_df['f1_malicious'], width=0.36, label='F1 malicious')
plt.bar([i + 0.18 for i in x], main_df['balanced_accuracy'], width=0.36, label='Balanced accuracy')
plt.xticks(list(x), main_df['method'], rotation=25, ha='right')
plt.ylim(0, 1.05)
plt.ylabel('Score')
plt.title('MIL baselines vs RC-PAA')
plt.legend()
plt.tight_layout()
fig1 = OUTPUT_DIR / 'figure_1_mil_baselines.png'
plt.savefig(fig1, dpi=200)
plt.show()

plt.figure(figsize=(10, 5))
plt.bar([i - 0.18 for i in x], main_df['FP'], width=0.36, label='False Positive')
plt.bar([i + 0.18 for i in x], main_df['FN'], width=0.36, label='False Negative')
plt.xticks(list(x), main_df['method'], rotation=25, ha='right')
plt.ylabel('Package count')
plt.title('Error breakdown by package-level method')
plt.legend()
plt.tight_layout()
fig2 = OUTPUT_DIR / 'figure_2_fp_fn.png'
plt.savefig(fig2, dpi=200)
plt.show()

print('A-G checklist:')
display(pd.DataFrame([{'group': k, **v} for k, v in output_checklist.items()]))
print('Saved outputs:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print(p.name)
